In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================

SILVER = "workspace.firmable_v2_silver"

INPUT_TABLE = f"{SILVER}.manager_candidates"
OUTPUT_TABLE = f"{SILVER}.manager_candidate_scores"

# ============================================================
# 2. READ CANDIDATES
# ============================================================

candidates = spark.table(INPUT_TABLE)

# ============================================================
# 3. SENIORITY GAP
# ============================================================

scored = candidates.withColumn(
    "seniority_gap",
    F.col("employee_seniority") -
    F.col("manager_seniority")
)

# ============================================================
# 4. STRUCTURAL RELATIONSHIP SCORE
# Direct relationships receive stronger evidence.
# Fallback relationships receive lower scores.
# ============================================================

scored = scored.withColumn(
    "relationship_fit_score",
    F.when(
        F.col("relationship_type").isin(
            "Team Lead to Individual Contributor",
            "Team Manager to Team Lead",
            "Department Head to Team Manager",
            "Functional Executive to Department Head",
            "Division President to Department Head",
            "CEO to C-Suite Executive",
            "CEO to Division President",
            "CEO to Department Head - Flat"
        ),
        45
    )
    .when(
        (F.col("relationship_type") == "Department Head to Team Lead") &
        (F.col("org_structure") == "Flat"),
        45
    )
    .when(
        F.col("relationship_type") == "Team Manager to Individual Contributor",
        24
    )
    .when(
        F.col("relationship_type") == "Department Head to Team Lead",
        22
    )
    .when(
        F.col("relationship_type") == "CEO to Department Head - Fallback",
        18
    )
    .when(
        F.col("relationship_type") == "Department Head to Individual Contributor",
        10
    )
    .otherwise(0)
)

# ============================================================
# 5. TEAM ALIGNMENT
# ============================================================

scored = scored.withColumn(
    "team_alignment_score",
    F.when(
        F.col("employee_team_id").isNotNull() &
        F.col("manager_team_id").isNotNull() &
        (
            F.col("employee_team_id") ==
            F.col("manager_team_id")
        ),
        15
    ).otherwise(0)
)

# ============================================================
# 6. DEPARTMENT ALIGNMENT
# ============================================================

scored = scored.withColumn(
    "department_alignment_score",
    F.when(
        F.col("employee_department").isNotNull() &
        F.col("manager_department").isNotNull() &
        (
            F.col("employee_department") ==
            F.col("manager_department")
        ),
        12
    ).otherwise(0)
)

# ============================================================
# 7. BUSINESS UNIT ALIGNMENT
# ============================================================

scored = scored.withColumn(
    "business_unit_alignment_score",
    F.when(
        F.col("employee_business_unit").isNotNull() &
        F.col("manager_business_unit").isNotNull() &
        (
            F.col("employee_business_unit") ==
            F.col("manager_business_unit")
        ),
        8
    ).otherwise(0)
)

# ============================================================
# 8. MANAGEMENT AUTHORITY
# ============================================================

scored = scored.withColumn(
    "authority_score",
    F.when(
        F.col("manager_authority") == "People Manager",
        8
    )
    .when(
        F.col("manager_authority") == "Potential People Manager",
        5
    )
    .when(
        F.col("manager_authority") == "Executive",
        4
    )
    .otherwise(0)
)

# ============================================================
# 9. SENIORITY GAP
# ============================================================

scored = scored.withColumn(
    "seniority_score",
    F.when(F.col("seniority_gap").between(1, 2), 5)
     .when(F.col("seniority_gap").between(3, 4), 3)
     .when(F.col("seniority_gap").between(5, 6), 2)
     .when(F.col("seniority_gap") > 6, 1)
     .otherwise(0)
)

# ============================================================
# 10. FINAL EVIDENCE SCORE
# ============================================================

scored = scored.withColumn(
    "candidate_score",
    F.least(
        F.lit(100),
        F.col("relationship_fit_score") +
        F.col("team_alignment_score") +
        F.col("department_alignment_score") +
        F.col("business_unit_alignment_score") +
        F.col("authority_score") +
        F.col("seniority_score")
    )
)

# ============================================================
# 11. RANK CANDIDATES
# ============================================================

ranking_window = (
    Window
    .partitionBy("company_id", "employee_id")
    .orderBy(
        F.desc("candidate_score"),
        F.asc("candidate_priority"),
        F.asc("seniority_gap"),
        F.asc("candidate_manager_id")
    )
)

scored = scored.withColumn(
    "candidate_rank",
    F.row_number().over(ranking_window)
)

# ============================================================
# 12. TOP AND SECOND-BEST SCORES
# ============================================================

employee_window = Window.partitionBy(
    "company_id",
    "employee_id"
)

scored = (
    scored
    .withColumn(
        "top_candidate_score",
        F.max(
            F.when(
                F.col("candidate_rank") == 1,
                F.col("candidate_score")
            )
        ).over(employee_window)
    )
    .withColumn(
        "second_candidate_score",
        F.max(
            F.when(
                F.col("candidate_rank") == 2,
                F.col("candidate_score")
            )
        ).over(employee_window)
    )
    .withColumn(
        "score_margin",
        F.when(
            F.col("second_candidate_score").isNotNull(),
            F.col("top_candidate_score") -
            F.col("second_candidate_score")
        )
    )
)

# ============================================================
# 13. AMBIGUITY FLAG
# ============================================================

scored = scored.withColumn(
    "ambiguity_flag",
    F.when(
        F.col("candidate_rank") == 1,
        F.when(
            F.col("score_margin").isNull(),
            False
        ).otherwise(
            F.col("score_margin") < 8
        )
    ).otherwise(False)
)

# ============================================================
# 14. PUBLISH SCORED CANDIDATES
# ============================================================

(
    scored.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

# ============================================================
# 15. INTEGRITY CHECKS
# ============================================================

total = scored.count()

unique_pairs = scored.select(
    "company_id",
    "employee_id",
    "candidate_manager_id"
).distinct().count()

invalid = scored.filter(
    (F.col("candidate_score") < 0) |
    (F.col("candidate_score") > 100) |
    (F.col("seniority_gap") <= 0)
).count()

assert total == unique_pairs, "Duplicate candidate pairs detected."
assert invalid == 0, "Invalid candidate scores detected."

# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("REFINED MANAGER CANDIDATE SCORING COMPLETED")

print("Total scored relationships:", total)

print("\nRELATIONSHIP SCORE DISTRIBUTION")

scored.groupBy(
    "relationship_type",
    "relationship_fit_score"
).count().orderBy(
    "relationship_type"
).show(50, truncate=False)

print("\nTOP CANDIDATE SCORE SUMMARY")

scored.filter(
    F.col("candidate_rank") == 1
).agg(
    F.count("*").alias("employees_with_candidates"),
    F.avg("candidate_score").alias("average_top_score"),
    F.min("candidate_score").alias("minimum_top_score"),
    F.max("candidate_score").alias("maximum_top_score"),
    F.avg("score_margin").alias("average_score_margin")
).show(truncate=False)

print("\nAMBIGUITY SUMMARY")

scored.filter(
    F.col("candidate_rank") == 1
).groupBy(
    "ambiguity_flag"
).count().show()

print("\nOUTPUT TABLE:", OUTPUT_TABLE)
print("ALL INTEGRITY CHECKS PASSED")